# 人生发财靠康波？— 量化实证检验

**Does Wealth Really Follow the Kondratieff Cycle?**

This notebook walks through five quantitative tests of Zhou Jintao's claim that asset markets follow a 60-year Kondratieff cycle with three specific "wealth windows" in living memory (1999, 2008, 2019).

**Author:** Yifan  
**License:** MIT  
**Companion article:** [Myth Buster #02](https://yifan70932.github.io/myth-busters/02-kangbo)

## Setup

In [ ]:
import sys
import os

# Add src/ to path
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, signal

from utils import (
    load_assets, load_gdp_growth,
    cagr, annual_returns, max_drawdown,
    candidate_years, control_years,
    ZHOU_WINDOWS, RANDOM_SEED
)

np.random.seed(RANDOM_SEED)
print(f'Zhou windows: {ZHOU_WINDOWS}')
print(f'Random seed: {RANDOM_SEED}')

## 1. Load all data

In [ ]:
assets = load_assets()
gdp = load_gdp_growth()

for name, prices in assets.items():
    print(f'{name:<20s} {prices.index.min()}-{prices.index.max()} ({len(prices)} obs)')
print(f'{"US GDP growth":<20s} {gdp.index.min()}-{gdp.index.max()} ({len(gdp)} obs)')

## 2. Test 1: Event Study (10-year CAGR)

If Zhou's windows are real opportunities, the windows should rank near the **top** of all candidate buy years. The reality:

In [ ]:
for asset_name, prices in assets.items():
    pairs = [(y, cagr(prices, y, 10)) for y in candidate_years(1995, 2014)]
    pairs = [(y, r) for y, r in pairs if not np.isnan(r)]
    pairs_sorted = sorted(pairs, key=lambda x: -x[1])
    rank_map = {y: i+1 for i, (y, _) in enumerate(pairs_sorted)}
    n = len(pairs)
    print(f'\n=== {asset_name} ===')
    for y in ZHOU_WINDOWS:
        if y in rank_map:
            r = next(r for yr, r in pairs if yr == y)
            print(f'  {y}: CAGR = {r:+6.2f}%, rank = {rank_map[y]}/{n}')
        else:
            print(f'  {y}: 10y horizon incomplete')
    best_y, best_r = pairs_sorted[0]
    worst_y, worst_r = pairs_sorted[-1]
    print(f'  Best:  {best_y} ({best_r:+.2f}%)')
    print(f'  Worst: {worst_y} ({worst_r:+.2f}%)')

**Key findings:**
- A-shares 2008: rank **20/20** (worst possible)
- S&P 500 1999: rank **20/20** (worst possible)
- No window ranks in the top 3 for any asset

## 3. Test 2: Statistical Inference

Three independent tests of whether window returns differ from the baseline:

In [ ]:
from scipy import stats as sp_stats

def bootstrap_p(treated, control, n_iter=10000, seed=42):
    rng = np.random.default_rng(seed)
    obs = np.mean(treated) - np.mean(control)
    pooled = np.array(treated + control)
    n_t = len(treated)
    diffs = np.empty(n_iter)
    for i in range(n_iter):
        perm = rng.permutation(pooled)
        diffs[i] = np.mean(perm[:n_t]) - np.mean(perm[n_t:])
    return np.mean(np.abs(diffs) >= np.abs(obs))

results = []
for asset_name, prices in assets.items():
    treated = [cagr(prices, y, 10) for y in ZHOU_WINDOWS]
    treated = [x for x in treated if not np.isnan(x)]
    control = [cagr(prices, y, 10) for y in control_years(ZHOU_WINDOWS, 1995, 2014)]
    control = [x for x in control if not np.isnan(x)]
    if not treated or not control:
        continue
    diff = np.mean(treated) - np.mean(control)
    _, p_t = sp_stats.ttest_ind(treated, control, equal_var=False)
    _, p_u = sp_stats.mannwhitneyu(treated, control, alternative='two-sided')
    p_b = bootstrap_p(treated, control)
    results.append({'asset': asset_name, 'Δ (pp)': round(diff, 2),
                    't-test p': round(p_t, 4),
                    'Mann-Whitney p': round(p_u, 4),
                    'Bootstrap p': round(p_b, 4)})

pd.DataFrame(results).set_index('asset')

**All 12 p-values > 0.05.** Cannot reject the null hypothesis that windows are indistinguishable from random buy years.

Note: All four Δ values are *negative* — windows actually underperform the baseline, which is the opposite of what the myth claims.

## 4. Test 3: Sharpe Ratio (risk-adjusted returns)

In [ ]:
RF = 3.5  # 10-year US Treasury long-run average

def sharpe(rets, rf=RF):
    if len(rets) < 2: return np.nan
    s = np.std(rets, ddof=1)
    return (np.mean(rets) - rf) / s if s > 0 else np.nan

rows = []
for asset_name, prices in assets.items():
    row = {'asset': asset_name}
    for y in [1999, 2008]:
        rets = annual_returns(prices, y, 10)
        row[str(y)] = round(sharpe(rets), 3) if len(rets) >= 5 else None
    bs = []
    for y in control_years(ZHOU_WINDOWS, 1995, 2014):
        rets = annual_returns(prices, y, 10)
        s = sharpe(rets)
        if not np.isnan(s): bs.append(s)
    row['baseline'] = round(np.mean(bs), 3) if bs else None
    rows.append(row)
pd.DataFrame(rows).set_index('asset')

## 5. Test 4: Maximum Drawdown

If windows are "low-point entries" as Zhou claims, max drawdown after the window should be **smaller** than baseline:

In [ ]:
rows = []
for asset_name, prices in assets.items():
    row = {'asset': asset_name}
    for y in ZHOU_WINDOWS:
        dd = max_drawdown(prices, y, 10)
        row[str(y)] = round(dd, 1) if not np.isnan(dd) else None
    baseline_dds = [max_drawdown(prices, y, 10) for y in control_years(ZHOU_WINDOWS, 1995, 2014)]
    baseline_dds = [d for d in baseline_dds if not np.isnan(d)]
    row['baseline'] = round(np.mean(baseline_dds), 1) if baseline_dds else None
    rows.append(row)
pd.DataFrame(rows).set_index('asset')

**6 of 8 windows show drawdowns *worse* than baseline** — the opposite of "low-point entry."

## 6. Test 5: Spectral Analysis of US GDP

Does a 60-year Kondratieff cycle exist in 155 years of US GDP growth?

In [ ]:
arr = gdp.values.astype(float)
detrended = signal.detrend(arr)
freqs, psd = signal.welch(detrended, fs=1.0, nperseg=60, scaling='density')
periods = 1.0 / freqs[1:]
psd = psd[1:]

# Bootstrap p-value at 60-year frequency
rng = np.random.default_rng(42)
target_idx = np.argmin(np.abs(periods - 60))
observed = psd[target_idx]
boot = []
for _ in range(5000):
    sh = rng.permutation(detrended)
    _, p_b = signal.welch(sh, fs=1.0, nperseg=60, scaling='density')
    boot.append(p_b[1:][target_idx])
boot = np.array(boot)
p_60y = np.mean(boot >= observed)

fig, ax = plt.subplots(figsize=(11, 5), dpi=110)
ax.plot(periods, psd, color='#1e3a5f', linewidth=1.8)
ax.fill_between(periods, psd, alpha=0.2, color='#1e3a5f')
ax.axhline(np.percentile(boot, 95), color='#2d5a3d', linestyle='--', linewidth=1, label='95% threshold')
ax.axvspan(50, 65, alpha=0.08, color='#1e3a5f', label='K-wave zone')
ax.set_xscale('log')
ax.set_xticks([2, 8.6, 20, 30, 50, 80])
ax.set_xticklabels([2, '8.6', 20, 30, 50, 80])
ax.set_xlabel('Period length (years, log scale)')
ax.set_ylabel('Power spectral density')
ax.set_title(f'US GDP growth power spectrum (1870-2024) — Bootstrap p at 60y = {p_60y:.3f}')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nBootstrap p-value at 60-year frequency: {p_60y:.4f}')
print(f'Conclusion: {"NOT significant" if p_60y > 0.05 else "Significant"}')

## 7. Combined verdict

All five independent tests fail to support Zhou's claim:

| Test | Result |
|---|---|
| **CAGR ranks** | A-shares 2008 = rank 20/20; S&P 1999 = rank 20/20 |
| **Statistical p-values** | All 12 p-values > 0.05 |
| **Sharpe ratios** | 6 of 8 windows below baseline |
| **Max drawdowns** | 6 of 8 windows worse than baseline |
| **GDP spectrum** | No 60-year peak (p = 0.69) |

**Final conclusion:** Zhou Jintao's specific "wealth windows" are not actionable investment signals. The popular slogan "人生发财靠康波" rests on retroactive pattern-recognition in too few historical observations to support its claims.